# LELA60341: Research Methods in Computational and Corpus Linguistics Week 8

## A `LogisticRegression` class

In the LELA60331 seminars you have built **logistic regression** step by step, as a long series of notebook cells, and in Week 7 you trained it on simulated data. In this workbook you will build the same classifier as a **class**, using what you learned about classes in Week 7. It works on numeric features, like the simulated data.

By the end of this workbook you will be able to write:

```python
model = LogisticRegression(lr=0.5, n_epochs=1000)
model.train(X, y)
print(model.evaluate(X, y))
```

| Part | Topic |
|---|---|
| 1 | From a script to a class |
| 2 | The data as a matrix |
| 3 | The `LogisticRegression` class |
| 4 | Training on the simulated data |
| 5 | Several models side by side: the learning rate |

Read each explanation, run the example cells, then try the 🏋️ exercises. In an exercise, replace each `???` with your own code, then run the ✅ check cell underneath. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell. Parts marked ⭐ **Challenge** are optional.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used by every exercise.
def check(label, condition):
    """Print a tick or a cross instead of crashing, so you can keep working."""
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 1 · From a script to a class

In the LELA60331 seminars, each step of the classifier is a separate cell, and the cells share their results through **global names**. Even for the simulated data in LELA60331 Week 7 there are a lot of them:

| Name | What it holds |
|---|---|
| `x`, `y`, `X` | the features and the labels |
| `weights`, `bias` | the model's parameters |
| `lr`, `n_epochs` | the training settings |
| `q`, `errors`, `loss_history` | the probabilities, the errors and the losses |
| `score`, `sigmoid`, `average_loss`, `mat_vec`, ... | the functions that work on them |

This works, but it is fragile:

* Run the cells in the wrong order, or re-run one by mistake, and the names hold the wrong values. (In LELA60331 Week 7, `q` holds the probabilities from whichever cell you ran last: the untrained model, the trained one, or one with a different learning rate.)
* You can't easily have **two** models at once, say one trained with each learning rate, because they would both want the name `weights`.
* To use the classifier on new data, you have to remember which cells to copy, and in which order.

A class solves all three problems. It bundles the **data** (as instance variables) together with the **operations** on it (as methods), as in Week 7. Each model is then one object, with its own settings and its own weights, and the rest of the program only needs to know what its methods do.

### 🏋️ Exercise 1 · Practice — Data or method?

The classifier will need each of the things below. For each one, say whether it should be stored as an `"instance variable"` or written as a `"method"`.

In [ ]:
design = {
    "the weights": ???,
    "calculate precision and recall": ???,
    "the learning rate": ???,
    "the probability that each example is positive": ???,
    "the list of losses, one per epoch": ???,
    "one step of gradient descent": ???,
}

In [ ]:
# ✅ Check
check("data", lambda: design["the weights"] == design["the learning rate"] == design["the list of losses, one per epoch"] == "instance variable")
check("operations", lambda: design["calculate precision and recall"] == design["the probability that each example is positive"] == design["one step of gradient descent"] == "method")

---
# Part 2 · The data as a matrix

Here is the simulated data from LELA60331 Weeks 5 and 7: 50 texts, each with two features (the log counts of positive and of negative words) and a label (1 = positive, 0 = negative). As in the seminar, `x[0]` holds feature 1 for every text and `x[1]` holds feature 2. Run the cell.

In [ ]:
import math
import random
import matplotlib.pyplot as plt

random.seed(10)
x = [[], []]
y = []
for i in range(50):
    if random.random() > 0.5:          # a negative text: fewer positive than negative words
        x[0].append(random.gauss(2, 1))
        x[1].append(random.gauss(3, 1))
        y.append(0)
    else:                              # a positive text: more positive than negative words
        x[0].append(random.gauss(3, 1))
        x[1].append(random.gauss(2, 1))
        y.append(1)
print(len(y), "texts,", sum(y), "of them positive")

In LELA60331 Week 7, Part 5 you saw that it is more convenient to store the data as a **matrix** `X` with **one row per text** and one column per feature, so that one matrix-vector multiplication gives the score of every text. You also saw that the gradients need the **transpose** of `X`: the same numbers with the rows turned into columns, one row per feature.

Our class will take its data in the form `X`, one row per example, so that it works for any number of features.

### 🏋️ Exercise 2 · Practice — Rows and the transpose

1. `X`: a list with one row `[x1, x2]` for each text (a list comprehension, as in LELA60331 Week 7, Activity 6)
2. Complete `transpose(matrix)`, which returns a new matrix whose row `j` is column `j` of `matrix`. *Hint:* column `j` is `[row[j] for row in matrix]` (Week 4, Part 3), and there is one column for each element of the first row.

In [ ]:
X = ???                  # 1.

def transpose(matrix):
    return ???   # 2.

print(len(X), len(X[0]), X[0])
print(transpose([[1, 2, 3], [4, 5, 6]]))       # you should get [[1, 4], [2, 5], [3, 6]]

In [ ]:
# ✅ Check
check("1. X has 50 rows of 2", lambda: len(X) == 50 and all(len(row) == 2 for row in X) and X[0] == [x[0][0], x[1][0]])
check("2. transpose of a small matrix", lambda: transpose([[1, 2, 3], [4, 5, 6]]) == [[1, 4], [2, 5], [3, 6]])
check("2. the transpose of X is x", lambda: transpose(X) == x)

---
# Part 3 · The `LogisticRegression` class

Two helper functions from LELA60331 Week 7 do the arithmetic: `dot(a, b)` and `mat_vec(matrix, vector)` (Activity 7). Run the cell; it also defines the `sigmoid` function from LELA60331 Week 5.

In [ ]:
def dot(a, b):
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # the dot product of each row of the matrix with the vector
    return [dot(row, vector) for row in matrix]

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

Here is what each part of the class does. Read through all of it, then complete the class in Exercise 3.

| Instance variable | What it holds |
|---|---|
| `_lr`, `_n_epochs`, `_seed` | the training settings: learning rate, number of epochs, and the random seed for the starting weights |
| `_weights`, `_bias` | the parameters |
| `_loss_history` | the loss after each epoch |

**`__init__(self, lr=0.5, n_epochs=1000, seed=10)`.** `lr=0.5` gives the parameter a **default value**: if you write `LogisticRegression()`, `lr` is 0.5, and if you write `LogisticRegression(lr=0.1)` it is 0.1. The defaults are the settings from LELA60331 Week 7. Before training there are no weights yet, so `_weights` is an empty list.

**Four small internal methods.** Their names start with an underscore, because they are for use inside the class (Week 7, *encapsulation*). Each one does a single step that depends on **how the features are stored**: here, as a list of rows. Keeping these steps in methods of their own means that a subclass that stores its features differently only needs to override these four methods, and can inherit everything else.

* `_num_features(X)`: the number of features, which is the length of a row
* `_start(num_features)`: sets the starting values, as in LELA60331 Week 7: `random.seed(self._seed)`, then a random weight between 0 and 1 for each feature, and a bias of 0
* `_scores(X)`: the score of every example (without the bias): `mat_vec` with `X` and the weights
* `_gradients(X, errors)`: for each feature, the dot product of that feature's values with the errors (before dividing by the number of examples): `mat_vec` with the **transpose** of `X` and the errors (LELA60331 Week 7, Activity 8)

**`predict_proba(X)`** returns the probability that each example is positive: the sigmoid of its score plus the bias.

**`predict(X)`** returns the predicted class of each example: 1 if its probability is above 0.5, and 0 otherwise. *Hint:* `int(p > 0.5)`.

**`average_loss(q, y)`** is the average cross-entropy loss from LELA60331 Week 5 (done for you). It adds a tiny number, `eps`, inside the log, so that it never takes the log of 0. That makes no visible difference here, but it matters when the scores are very large.

**`train(X, y)`** is the training loop from LELA60331 Week 7, Activity 1: set the starting values, then on each epoch calculate the probabilities, record the loss, calculate the errors and gradients, and update the weights and the bias. Everything the loop needs to keep is an instance variable: `self._weights`, not `weights`. Notice that it never mentions how many features there are: `_gradients` gives one gradient per weight, however many there are. (It calculates the probabilities with the same line as `predict_proba`, rather than calling `predict_proba` itself, so that a subclass can change what `predict_proba` takes as input without breaking `train`.)

**`evaluate(X, y)`** counts the correct predictions, true positives, false positives and false negatives (done for you), and returns a **dict** (Week 2) with three entries: `"accuracy"`, `"precision"` and `"recall"` (LELA60331 Week 7, Activities 2 and 5).

**`loss_history()`** and **`parameters()`** give the user the losses (for plotting) and the weights and bias, without touching the instance variables directly.

### 🏋️ Exercise 3 · Practice — Complete `LogisticRegression`

Fill in the blanks 1–8. The cell won't run until every `???` is replaced, so work through them in order. The tests underneath train a model on the first 10 texts only.

In [ ]:
class LogisticRegression:
    """Logistic regression for examples given as rows of numeric features."""

    def __init__(self, lr=0.5, n_epochs=1000, seed=10):
        self._lr = lr
        self._n_epochs = n_epochs
        self._seed = seed
        self._weights = []
        self._bias = 0
        self._loss_history = []

    # ---------------------------------------------------------------- steps that depend on how the features are stored
    def _num_features(self, X):
        return len(X[0])

    def _start(self, num_features):
        random.seed(self._seed)
        self._weights = [random.random() for j in range(num_features)]
        self._bias = 0

    def _scores(self, X):
        return ???                          # 1.

    def _gradients(self, X, errors):
        return ???                      # 2.

    # ---------------------------------------------------------------- prediction
    def predict_proba(self, X):
        """The probability that each example is positive."""
        return ???     # 3.

    def predict(self, X):
        """The predicted class (1 or 0) of each example."""
        return ???     # 4.

    # ---------------------------------------------------------------- training
    def average_loss(self, q, y):
        eps = 0.00001          # a tiny number, so that we never take the log of 0
        losses = []
        for k in range(len(y)):
            if y[k] == 1:
                losses.append(-math.log2(q[k] + eps))
            else:
                losses.append(-math.log2(1 - q[k] + eps))
        return sum(losses) / len(losses)

    def train(self, X, y, verbose=False):
        """Learn the weights and bias from the rows X and their labels y (1 or 0)."""
        num_samples = len(y)
        self._start(self._num_features(X))
        self._loss_history = []
        for epoch in range(self._n_epochs):
            q = [sigmoid(value + self._bias) for value in self._scores(X)]    # as in predict_proba
            self._loss_history.append(self.average_loss(q, y))
            errors = ???                      # 5.
            dw = ???       # 6.
            db = ???
            self._weights = ???   # 7.
            self._bias = ???
            if verbose and epoch % 50 == 0:
                print("epoch", epoch, " loss", round(self._loss_history[-1], 3))

    def loss_history(self):
        return self._loss_history

    def parameters(self):
        return self._weights, self._bias

    # ---------------------------------------------------------------- evaluation
    def evaluate(self, X, y):
        """A dict with the accuracy, precision and recall of the model on X."""
        predicted = self.predict(X)
        correct = 0
        tp = 0
        fp = 0
        fn = 0
        for k in range(len(y)):
            if predicted[k] == y[k]:
                correct += 1
            if predicted[k] == 1 and y[k] == 1:
                tp += 1
            if predicted[k] == 1 and y[k] == 0:
                fp += 1
            if predicted[k] == 0 and y[k] == 1:
                fn += 1
        return {"accuracy": ???,                   # 8.
                "precision": ???,
                "recall": ???}

    # ⭐ Challenge: __str__
    # def __str__(self):
    #     return ???


small = LogisticRegression(lr=0.5, n_epochs=100)
small.train(X[:10], y[:10])
print(small.parameters())
print(small.predict_proba(X[:3]), small.predict(X[:10]))
print(small.evaluate(X[:10], y[:10]))      # you should get an accuracy of 0.9, precision 1.0 and recall 0.667

In [ ]:
# ✅ Check
def _start_values():
    random.seed(10)
    return [random.random(), random.random()]
_w, _b = small.parameters()
_m = LogisticRegression()
_m._start(2)
check("the starting values", lambda: _m.parameters() == (_start_values(), 0))
check("1. _scores", lambda: small._scores([[1, 0], [0, 1], [2, 3]]) == [_w[0], _w[1], 2 * _w[0] + 3 * _w[1]])
check("2. _gradients", lambda: small._gradients([[1, 2], [3, 4]], [10, 1]) == [13, 24])
check("3. predict_proba", lambda: abs(small.predict_proba([[1, 2]])[0] - sigmoid(_b + _w[0] + 2 * _w[1])) < 1e-12)
check("4. predict", lambda: small.predict(X[:10]) == [int(p > 0.5) for p in small.predict_proba(X[:10])])
check("5–7. the loss went down", lambda: len(small.loss_history()) == 100 and small.loss_history()[-1] < small.loss_history()[0])
check("5–7. training changed the starting weights", lambda: len(_w) == 2 and _w != _start_values())
check("8. evaluate", lambda: small.evaluate(X[:10], y[:10]) == {"accuracy": 0.9, "precision": 1.0, "recall": 2 / 3})

---
# Part 4 · Training on the simulated data

### 🏋️ Exercise 4 · Practice — LELA60331 Week 7, as an object

1. Make `model`, a `LogisticRegression` with the default settings, and train it on all of `X` and `y`.
2. Plot its loss history.
3. `results`: the result of evaluating it on `X` and `y`
4. `new_probs`: the probability that each of these two new texts is positive (LELA60331 Week 7, Activity 4). Remember that the features are **log** counts.
   * a) 2 positive words and 3 negative words
   * b) 10 positive words and 1 negative word

In [ ]:
model = ???                                         # 1.
???

plt.plot(???)                                       # 2.
plt.xlabel("epoch")
plt.ylabel("loss")
plt.show()

results = ???                                       # 3.
new_probs = ???   # 4.
print(model.parameters())        # about [1.17, -1.32] and 0.51
print(results)
print(new_probs)                 # about 0.466 and 0.961

In [ ]:
# ✅ Check
check("1. the same weights as LELA60331 Week 7", lambda: [round(w, 2) for w in model.parameters()[0]] == [1.17, -1.32] and round(model.parameters()[1], 2) == 0.51)
check("1. final loss about 0.656", lambda: abs(model.loss_history()[-1] - 0.656) < 0.001)
check("3. accuracy 0.78, precision 0.75, recall 0.783",
      lambda: results["accuracy"] == 0.78 and results["precision"] == 0.75 and abs(results["recall"] - 0.783) < 0.001)
check("4. new_probs", lambda: abs(new_probs[0] - 0.466) < 0.005 and abs(new_probs[1] - 0.961) < 0.005)

These are the numbers you got in LELA60331 Week 7. Compare the two pieces of code: what does the class do for you that the seminar's cells made you do by hand?

---
# Part 5 · Several models side by side: the learning rate

In LELA60331 Week 7, Activity 3, you experimented with the learning rate by changing `lr` and re-running the training cell, which replaced the old model each time. With a class, each model is a separate object, so you can keep several and compare them.

### 🏋️ Exercise 5 · Practice — Comparing learning rates

1. `models`: a dict (Week 2) from each learning rate in `rates` to a new `LogisticRegression` with that learning rate (and the default 1000 epochs). *Hint:* a loop that adds one entry per learning rate.
2. Train each model on `X` and `y`.

The cell then plots the three loss histories on the same axes.

🔮 **Predict** before you run it: what will each curve look like? (Look back at your notes from LELA60331 Week 7.)

In [ ]:
rates = [0.01, 0.5, 2]
models = {}
for lr in rates:
    models[lr] = ???                             # 1.
    ???                                             # 2.

for lr in rates:
    plt.plot(models[lr].loss_history(), label="lr = " + str(lr))
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()
for lr in rates:
    print(lr, round(models[lr].loss_history()[-1], 3), models[lr].evaluate(X, y))

In [ ]:
# ✅ Check
check("1. three models", lambda: sorted(models) == rates and all(isinstance(m, LogisticRegression) for m in models.values()))
check("1. each with its own learning rate", lambda: [models[lr]._lr for lr in rates] == rates)
check("2. each trained", lambda: all(len(m.loss_history()) == 1000 for m in models.values()))
check("the first model is unchanged", lambda: abs(model.loss_history()[-1] - 0.656) < 0.001)

⭐ **Challenge (optional):** add a method `__str__` to the class, so that `print(model)` shows its settings, for example `LogisticRegression(lr=0.5, n_epochs=1000)`. Then print the three models in `models`.

---
## ✏️ Summary

You can now:

* explain why a program with many shared global names is fragile, and how a class helps
* design a class by deciding what is **data** (instance variables) and what is an **operation** (methods)
* give a parameter a **default value**
* write a `LogisticRegression` class that trains, predicts and evaluates, keeping everything it needs between calls in instance variables
* keep the steps that depend on how the data is stored in small internal methods of their own
* keep several objects of the same class side by side, for example to compare settings